In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import math
import copy

import matplotlib.pyplot as plt
from data_loader import *
from utils import *
from utils_draw import *
from GAN import *
from GAN_joint import *
from combined_data_loader import *
from Optimization_multi_node import *
from Optimization_single_node import *
from tqdm import tqdm

In [2]:
class Args:
    def __init__(self):
        # ----- problem / optnet -----
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 4.5
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4
        self.pwl_segments = 10

        # IMPORTANT: add these to match gurobi
        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        self.N_scen = 10       # <== OptNet真正求解的场景池 (即 K)
        self.S_full = 100       # VAE 现场吐出的大量候选场景数 (S 池)
        self.K_rand = 0       # K里面有多少条纯随机保留(防过拟合)
        self.tau_gumbel = 1.0     # Gumbel Softmax 温度
        self.eps_uniform = 0.1 # 防震荡平滑参数
        self.lambda_div = 1e5   # [新增] 避免多头选到同一个场景的相互排斥惩罚力度

        # ----- 分段训练控制参数 -----
        self.device = "cuda"
        self.batch_size = 32   # (注意: 原来你的叫 dfl_batch_size, 统一改成 batch_size 给 DataLoader 读)
        self.dfl_batch_size = 8
        self.solver = "ECOS"
        
        self.filter_epochs = 5 # Stage 2 (训Filter) 轮数
        self.filter_lr = 1e-3   # Stage 2 学习率
        self.dfl_epochs = 1     # Stage 3 (联合微调) 轮数 (端到端微调极耗时，一般1-3轮即收敛)
        self.dfl_lr = 1e-6      # Stage 3 学习率 (必须极小，防崩坏)
        
args = Args()

In [3]:

set_seed(42)
data_path = "load_data_city_4_2.csv"
eps_search=pd.read_csv('../Result/eps_search.csv')
eps=int(eps_search[eps_search['model']=='gan_m']['eps'])
target_nodes = [f"4-2-{i}" for i in range(11)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data=pd.read_csv(data_path)
data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
data_2022 = data[data["DATETIME"].dt.year == 2022].copy()
Lmin, Lmax = system_hourly_load_minmax(data_2022, datetime_col="DATETIME",node_cols=target_nodes)
Lmax_total=Lmax.sum(0)# (24,)
Lmin_total=Lmin.sum(0) # (24,)
args.Lmax_total=Lmax_total
args.Lmin_total=Lmin_total
args.eps_value=eps

In [6]:

# set_seed(42)
# data_path = "load_data_city_4_2.csv"
# target_nodes = [f"4-2-{i}" for i in range(11)]
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_m, handlers_m, pack_data_m = run_gan_joint(
    data_path=data_path,
    node_cols=target_nodes,
    device=device,

    # train
    epochs=1000,
    batch_size=128,
    patience=50,
    seed=42,
    train_length=8760,
    val_ratio=0.2,
    add_lag1=True,

    # optim
    lr_g=1e-4,
    lr_d=4e-4,
    n_critic=1,
    grad_clip=1.0,

    # losses
    lambda_adv=1.0,
    lambda_sup=10.0,
    sup_samples=16,
    sup_warmup_epochs=50,
    sup_loss_type="l1",   # "l1" | "huber" | "mse"

    # model
    z_global_dim=128,
    z_node_dim=8,
    trunk_hidden=512,
    trunk_blocks=3,
    head_hidden=512,
    head_blocks=3,
    dropout=0.0,

    val_n_samples=50,
    val_z_temp=1.0,
    val_share_znode=False,
    val_quantiles=[0.05 * i for i in range(1, 20)],

    find_best_ztemp=True,
    z_temp_grid=[0.5*i for i in range(1,9)],
    ztemp_search_n_samples=50,
    ztemp_search_batch_size=128,
    ztemp_share_znode=None,
    ckpt_dir="../Model/GAN/ckpt_joint_gan",
    verbose=True,
)


[ep 0001] train D=1.581948 G_adv=0.160097 SUP=0.784578 | val pinball(norm)=0.371838
[ep 0010] train D=1.089955 G_adv=1.202417 SUP=0.589214 | val pinball(norm)=0.240008
[ep 0020] train D=1.895737 G_adv=0.222588 SUP=0.281659 | val pinball(norm)=0.119776
[ep 0030] train D=1.838878 G_adv=0.185976 SUP=0.221733 | val pinball(norm)=0.098993
[ep 0040] train D=1.724656 G_adv=0.621854 SUP=0.219262 | val pinball(norm)=0.099140
[ep 0050] train D=1.716294 G_adv=0.380934 SUP=0.202144 | val pinball(norm)=0.094241
[ep 0060] train D=1.670091 G_adv=0.617901 SUP=0.209069 | val pinball(norm)=0.091569
[ep 0070] train D=1.672946 G_adv=0.465877 SUP=0.201131 | val pinball(norm)=0.092113
[ep 0080] train D=1.863459 G_adv=0.335903 SUP=0.198219 | val pinball(norm)=0.088103
[ep 0090] train D=1.705788 G_adv=0.430504 SUP=0.198125 | val pinball(norm)=0.086822
[ep 0100] train D=1.788476 G_adv=0.334614 SUP=0.195219 | val pinball(norm)=0.088150
[ep 0110] train D=1.764100 G_adv=0.405211 SUP=0.194129 | val pinball(norm)=0

In [ ]:

set_seed(0)
window_pack_m_gan_train = sample_window_gan_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=292, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="train",
)

set_seed(0)
window_pack_m_gan_val = sample_window_gan_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=73, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="val",
)

set_seed(0)
window_pack_m_gan_test = sample_window_gan_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=303, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="test",
)


window_pack_draw = sample_window_gan_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=3, start_day=0, n_samples=200, seq_len=24,
    z_temp=None, split="test",
)


dfm = compute_metrics_window(window_pack_m_gan_test)
print(dfm)

print("best_z_temp:", pack_data_m["_JOINT_"]["best_z_temp"])
print("best_val_pinball_real:", pack_data_m["_JOINT_"]["best_val_pinball_real"])
print("ztemp_table:", pack_data_m["_JOINT_"]["ztemp_table"])

      node     L         mse       rmse  pinball_avg
0    4-2-0  7272   40.088486   6.331547     1.810294
1    4-2-1  7272  136.460419  11.681627     3.297946
2    4-2-2  7272   69.024971   8.308127     2.345342
3    4-2-3  7272  106.615166  10.325462     2.984669
4    4-2-4  7272   54.184952   7.361043     2.167328
5    4-2-5  7272   75.902077   8.712180     2.512276
6    4-2-6  7272   52.213428   7.225886     2.069997
7    4-2-7  7272   59.124855   7.689269     2.167756
8    4-2-8  7272   74.504639   8.631607     2.547916
9    4-2-9  7272   64.723663   8.045102     2.294211
10  4-2-10  7272   16.595314   4.073735     1.208580
best_z_temp: 1.5
best_val_pinball_real: 2.1480915546417236
ztemp_table: [(0.5, 2.391859769821167), (1.0, 2.1968135833740234), (1.5, 2.1480915546417236), (2.0, 2.22287654876709), (2.5, 2.3804492950439453), (3.0, 2.6152684688568115), (3.5, 2.9079554080963135), (4.0, 3.218858480453491)]


In [23]:
import pickle

with open('../Result/GAN/models_m.pkl', 'wb') as f:
    pickle.dump(models_m, f)
with open('../Result/GAN/handlers_m.pkl', 'wb') as f:
    pickle.dump(handlers_m, f)
with open('../Result/GAN/pack_data_m.pkl', 'wb') as f:
    pickle.dump(pack_data_m, f)

with open('../Result/GAN/window_pack_m_gan_test.pkl','wb') as f:
    pickle.dump(window_pack_m_gan_test, f)

with open('../Result/GAN/window_pack_m_gan_train.pkl','wb') as f:
    pickle.dump(window_pack_m_gan_train, f)

with open('../Result/GAN/window_pack_m_gan_val.pkl','wb') as f:
    pickle.dump(window_pack_m_gan_val, f)